# spaDIVA tutorial: simulated spatial multi-omics data

This tutorial runs spaDIVA on a small paired spatial multi-omics example included in the release. It demonstrates the standard workflow: load data, preprocess modalities, build a spatial graph, train spaDIVA, infer latent representations, and inspect shared and modality-specific outputs.


## 1. Import packages

Install spaDIVA from the repository root before running the tutorials: pip install -e . --no-deps.


In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import torch
from sklearn.decomposition import PCA
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
from spaDIVA.analysis import fuse_shared_latent, collect_spadiva_outputs
from spaDIVA.train import train_spadiva, infer_latents
from spaDIVA.utils import cal_spatial, clr_normalize_each_cell, cluster

sc.set_figure_params(figsize=(3, 3))
plt.rcParams["figure.dpi"] = 120


## 2. Set run options

`USE_CUDA` is set to `False` for a portable quick start. If your PyTorch installation is configured for your GPU, you can set it to `True`.


In [ ]:
RANDOM_SEED = 42
POE_SAMPLE_SEED = RANDOM_SEED
USE_CUDA = False
MAX_EPOCHS = 1000

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)


## 3. Load the example data

The example contains two paired modalities measured at the same simulated spatial locations.


In [ ]:
DATA_DIR = PROJECT_ROOT / "examples" / "simulation_data"
adata_omics1 = sc.read(DATA_DIR / "ADT_100.h5ad")
adata_omics2 = sc.read(DATA_DIR / "RNA_ZINB.h5ad")

adata_omics1.var_names_make_unique()
adata_omics2.var_names_make_unique()
adata_omics1.X = adata_omics1.layers["counts"].copy()
adata_omics2.X = adata_omics2.layers["counts"].copy()
adata_omics1.X = adata_omics1.X.astype("float32")
adata_omics2.X = adata_omics2.X.astype("float32")

adata_omics1, adata_omics2


## 4. Optional: prepare simulation-specific labels

These labels support evaluation in this simulated example; real-data applications proceed without ground-truth labels.


In [ ]:
A = adata_omics1.obs["category"]
B = adata_omics2.obs["category"]

ground_truth = pd.Series("Background", index=A.index)
ground_truth[(A != "Background") & (B == "Background")] = A[(A != "Background") & (B == "Background")]
ground_truth[(A == "Background") & (B != "Background")] = B[(A == "Background") & (B != "Background")]
ground_truth[(A != "Background") & (B != "Background")] = A[(A != "Background") & (B != "Background")]

Y = ground_truth.values
Y_shared = ground_truth.copy()
Y_shared[~Y_shared.isin(["factor 1", "factor 2"])] = "Background"
Y_shared = Y_shared.values

Y_omics1 = ground_truth.copy()
Y_omics1[Y_omics1 != "factor 3"] = "Background"
Y_omics1 = Y_omics1.values

Y_omics2 = ground_truth.copy()
Y_omics2[Y_omics2 != "factor 4"] = "Background"
Y_omics2 = Y_omics2.values

adata_omics1.obs["combined_truth"] = ground_truth.astype("category")
adata_omics2.obs["combined_truth"] = ground_truth.astype("category")


## 5. Visualize input labels


In [ ]:
palette = {
    "Background": "#1f77b4",
    "factor 1": "#ff7f0e",
    "factor 2": "#2ca02c",
    "factor 3": "#d62728",
    "factor 4": "#9467bd",
}

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
sc.pl.embedding(adata_omics1, basis="spatial", color="category", palette=palette, title="Omics 1", s=100, ax=axes[0], show=False)
sc.pl.embedding(adata_omics2, basis="spatial", color="category", palette=palette, title="Omics 2", s=100, ax=axes[1], show=False)
sc.pl.embedding(adata_omics1, basis="spatial", color="combined_truth", palette=palette, title="Combined truth", s=100, ax=axes[2], show=False)
plt.tight_layout()
plt.show()


## 6. Preprocess each modality

For this example, omics 1 is CLR-normalized and omics 2 is log-normalized. Both modalities are reduced to 64 principal components before training.


In [ ]:
adata_omics1 = clr_normalize_each_cell(adata_omics1)
sc.pp.scale(adata_omics1)

sc.pp.normalize_total(adata_omics2, target_sum=1e4)
sc.pp.log1p(adata_omics2)
sc.pp.scale(adata_omics2)

pca1 = PCA(n_components=64)
adata_omics1.obsm["pca"] = pca1.fit_transform(adata_omics1.to_df())

pca2 = PCA(n_components=64)
adata_omics2.obsm["pca"] = pca2.fit_transform(adata_omics2.to_df())


## 7. Build the spatial graph and training matrices

spaDIVA uses the spatial graph to pass information between neighboring spots.


In [ ]:
spatial = adata_omics1.obsm["spatial"]
edge_index = cal_spatial(spatial, k=4)

X1_input = adata_omics1.obsm["pca"]
X2_input = adata_omics2.obsm["pca"]
X1_train = X1_input
X2_train = X2_input

edge_index.shape, X1_input.shape, X2_input.shape


## 8. Train spaDIVA once


In [ ]:
model, train_loss = train_spadiva(
    X1_input,
    X2_input,
    X1_train,
    X2_train,
    edge_index=edge_index,
    learning_rate=1e-3,
    weight=1.0,
    max_epochs=MAX_EPOCHS,
    use_cuda=USE_CUDA,
)


## 9. Inspect training loss


In [ ]:
plt.plot(train_loss)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("spaDIVA training loss")
plt.show()


## 10. Infer and collect spaDIVA representations

spaDIVA returns modality-specific shared estimates (`Z1`, `Z2`) and modality-specific representations (`W1`, `W2`). The helper functions below fuse the shared estimates and collect the outputs into one `AnnData` object.


In [ ]:
Z_poe, Z1_loc, Z2_loc, W1_loc, W2_loc, X1_hat, X2_hat = infer_latents(
    model,
    X1_input,
    X2_input,
    edge_index=edge_index,
    use_cuda=USE_CUDA,
    sample_seed=POE_SAMPLE_SEED,
)

Z_loc = fuse_shared_latent(Z1_loc, Z2_loc, k=20)

z_adata = collect_spadiva_outputs(
    Z=Z_loc,
    W1=W1_loc,
    W2=W2_loc,
    spatial=spatial,
    obs_names=adata_omics1.obs_names,
    modality_names=("omics1", "omics2"),
    Z_poe=Z_poe,
    Z1=Z1_loc,
    Z2=Z2_loc,
    X1_hat=X1_hat,
    X2_hat=X2_hat,
    model_seed=RANDOM_SEED,
    poe_sample_seed=POE_SAMPLE_SEED,
)

z_adata


## 11. Cluster and visualize spaDIVA outputs

The integrated representation `Z_W` combines shared and modality-specific information. The `W_omics1` and `W_omics2` blocks can be inspected separately to visualize modality-specific structure.


In [ ]:
cluster_settings = {
    "Z": {"n_clusters": 3, "title": "Shared representation"},
    "Z_W": {"n_clusters": 5, "title": "Integrated representation"},
    "W_omics1": {"n_clusters": 2, "title": "Omics 1-specific representation"},
    "W_omics2": {"n_clusters": 2, "title": "Omics 2-specific representation"},
}

for key, cfg in cluster_settings.items():
    z_adata.obs[f"mclust_{key}"] = cluster(
        z_adata.obsm[key],
        num_cluster=cfg["n_clusters"],
        spatial=spatial,
        title=cfg["title"],
        random_seed=RANDOM_SEED,
        s=100,
        show=True,
        return_labels=True,
    )


## 12. Optional: evaluate with simulation labels

Because this dataset is simulated, we can compare the clusters with known labels. This evaluation step is specific to the simulated example.


In [ ]:
evaluation_targets = {
    "Z": Y_shared,
    "Z_W": Y,
    "W_omics1": Y_omics1,
    "W_omics2": Y_omics2,
}

metrics = []
for key, truth in evaluation_targets.items():
    pred = z_adata.obs[f"mclust_{key}"]
    metrics.append({
        "representation": key,
        "ARI": adjusted_rand_score(truth, pred),
        "NMI": normalized_mutual_info_score(truth, pred),
    })

pd.DataFrame(metrics)


## 13. Result object

The final `AnnData` object stores all spaDIVA representations in `.obsm`.


In [ ]:
z_adata
